In [1]:
# ============================================================
# 1. Imports and experiment settings
# ============================================================

from pathlib import Path
import ast
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# This is the directory structure used by the original notebook.
# Change this path if the result files are stored elsewhere.
import os


MAIN_DIR = Path('../../results/random_configurations')

DATASETS = ['D2', 'D3', 'D4', 'D8', 'D9', 'D10']
THRESHOLDS = [85, 90, 95]

METHOD_FILES = {
    'WEP': Path('metablocking') / '{dataset}' / 'WeightedEdgePruning.csv',
    'WNP': Path('metablocking') / '{dataset}' / 'WeightedNodePruning.csv',
    'CEP': Path('metablocking') / '{dataset}' / 'CardinalityEdgePruning.csv',
    'CNP': Path('metablocking') / '{dataset}' / 'CardinalityNodePruning.csv',
    'BLAST': Path('metablocking') / '{dataset}' / 'BLAST.csv',
}

METHOD_ORDER = ['WEP', 'WNP', 'CEP', 'CNP', 'BLAST']

In [2]:
# ============================================================
# 2. Load all experiment results
# ============================================================

def get_method_file(method, dataset):
    """Return the CSV path for a method and dataset."""
    return MAIN_DIR / METHOD_FILES[method].as_posix().format(dataset=dataset)


def load_results():
    """Load all result CSVs into a dictionary keyed by dataset and method."""
    data = {}
    missing = []

    for dataset in DATASETS:
        data[dataset] = {}

        for method in METHOD_ORDER:
            path = get_method_file(method, dataset)

            if not path.exists():
                missing.append(str(path))
                continue

            df = pd.read_csv(path)

            # Make sure the metrics used for selection are numeric.
            for col in ['Precision %', 'Recall %', 'F1 %', 'runtime',
                    'candidate_pairs', 'total_runtime']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            data[dataset][method] = df

    if missing:
        print('WARNING: The following result files were not found:')
        for path in missing:
            print('  ', path)

    return data


results = load_results()

print('Loaded result files:')
for dataset in DATASETS:
    for method in METHOD_ORDER:
        if method in results.get(dataset, {}):
            print(f'  {dataset:>3} | {method:<9} | {len(results[dataset][method]):,} configurations')

Loaded result files:
   D2 | WEP       | 1,500 configurations
   D2 | WNP       | 1,500 configurations
   D2 | CEP       | 1,500 configurations
   D2 | CNP       | 1,500 configurations
   D2 | BLAST     | 1,500 configurations
   D3 | WEP       | 1,500 configurations
   D3 | WNP       | 1,500 configurations
   D3 | CEP       | 1,500 configurations
   D3 | CNP       | 1,500 configurations
   D3 | BLAST     | 1,500 configurations
   D4 | WEP       | 1,500 configurations
   D4 | WNP       | 1,500 configurations
   D4 | CEP       | 1,500 configurations
   D4 | CNP       | 1,500 configurations
   D4 | BLAST     | 1,500 configurations
   D8 | WEP       | 1,500 configurations
   D8 | WNP       | 1,500 configurations
   D8 | CEP       | 1,500 configurations
   D8 | CNP       | 1,500 configurations
   D8 | BLAST     | 1,500 configurations
   D9 | WEP       | 1,500 configurations
   D9 | WNP       | 1,500 configurations
   D9 | CEP       | 1,500 configurations
   D9 | CNP       | 1,500 configurat

In [3]:
# ============================================================
# 3. Configuration-selection rule
# ============================================================

def select_configuration(df, threshold):
    """
    Select exactly one configuration according to the requested rule.

    Rule:
      1. Keep configurations with Recall % >= threshold.
      2. If any exist, choose highest precision, then highest recall.
      3. Otherwise choose highest recall, then highest precision,
         and mark the result as a fallback.
    """

    required = ['Precision %', 'Recall %']
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    candidates = df.dropna(subset=required).copy()

    if candidates.empty:
        return None

    # First choice: configurations that meet the requested recall.
    eligible = candidates[candidates['Recall %'] >= threshold]

    if not eligible.empty:
        selected = (
            eligible
            .sort_values(
                by=['Precision %', 'Recall %'],
                ascending=[False, False],
                kind='mergesort'
            )
            .iloc[0]
        )
        selection_type = 'Threshold met'
        fallback = False
    else:
        # No configuration reaches the target: maximize recall,
        # using precision as the secondary criterion.
        selected = (
            candidates
            .sort_values(
                by=['Recall %', 'Precision %'],
                ascending=[False, False],
                kind='mergesort'
            )
            .iloc[0]
        )
        selection_type = 'Fallback: max recall'
        fallback = True

    return {
        'row': selected,
        'selection_type': selection_type,
        'fallback': fallback,
    }

In [4]:
# ============================================================
# 4. Build the selected-results dataframe
# ============================================================

def build_selected_results(results):
    rows = []

    for dataset in DATASETS:
        for threshold in THRESHOLDS:
            for method in METHOD_ORDER:

                df = results.get(dataset, {}).get(method)

                if df is None:
                    rows.append({
                        'Dataset': dataset,
                        'Target Recall': threshold,
                        'Method': method,
                        'Selection': 'No result file',
                        'Fallback': True,
                        'Original Row Index': np.nan, # <--- Added
                        'Precision %': np.nan,
                        'Recall %': np.nan,
                        'Candidate pairs': np.nan,
                        'Runtime': np.nan,
                        'F1 %': np.nan,
                        'Configuration': None,
                        'Weighting Scheme': np.nan,
                        'Total Runtime': np.nan,
                    })
                    continue

                selection = select_configuration(df, threshold)

                if selection is None:
                    rows.append({
                        'Dataset': dataset,
                        'Target Recall': threshold,
                        'Method': method,
                        'Selection': 'No valid configuration',
                        'Fallback': True,
                        'Original Row Index': np.nan, # <--- Added
                        'Precision %': np.nan,
                        'Recall %': np.nan,
                        'Candidate pairs': np.nan,
                        'Runtime': np.nan,
                        'F1 %': np.nan,
                        'Configuration': None,
                        'Weighting Scheme': np.nan,
                        'Total Runtime': np.nan,
                    })
                    continue

                row = selection['row']

                rows.append({
                    'Dataset': dataset,
                    'Target Recall': threshold,
                    'Method': method,
                    'Selection': selection['selection_type'],
                    'Fallback': selection['fallback'],
                    'Original Row Index': row.name, # <--- GRABS ORIGINAL INDEX HERE
                    'Precision %': row.get('Precision %', np.nan),
                    'Recall %': row.get('Recall %', np.nan),
                    'Candidate pairs': row.get('candidate_pairs', np.nan),
                    'Runtime': row.get('runtime', np.nan),
                    'F1 %': row.get('F1 %', np.nan),
                    'Configuration': row.get('config', None),
                    'Weighting Scheme': row.get('weighting_scheme', np.nan),
                    'Total Runtime': row.get('total_runtime', row.get('runtime', np.nan)),
                })


    out = pd.DataFrame(rows)

    out['Method'] = pd.Categorical(
        out['Method'], categories=METHOD_ORDER, ordered=True
    )
    out['Dataset'] = pd.Categorical(
        out['Dataset'], categories=DATASETS, ordered=True
    )

    return out.sort_values(
        ['Target Recall', 'Dataset', 'Method']
    ).reset_index(drop=True)


selected_results = build_selected_results(results)

display(selected_results[
    ['Dataset', 'Target Recall', 'Method', 'Selection', 'Original Row Index',
     'Precision %', 'Recall %', 'Candidate pairs', 'Runtime', 'Total Runtime', ]
].head(12))


,Dataset,Target Recall,Method,Selection,Original Row Index,Precision %,Recall %,Candidate pairs,Runtime,Total Runtime
0,D2,85,WEP,Threshold met,509,12.35,85.15,7337,0.09,1.29
1,D2,85,WNP,Threshold met,851,8.49,85.15,10673,0.28,1.12
2,D2,85,CEP,Threshold met,315,3.40,85.15,26640,0.03,0.87
3,D2,85,CNP,Threshold met,324,3.50,85.15,25866,0.25,1.09
4,D2,85,BLAST,Threshold met,244,58.65,85.06,1543,0.49,1.50
5,D3,85,WEP,Threshold met,812,3.65,86.39,26108,0.07,2.21
6,D3,85,WNP,Threshold met,824,2.59,85.93,36615,0.37,2.51
7,D3,85,CEP,Threshold met,813,1.16,89.29,85053,0.10,2.24
8,D3,85,CNP,Threshold met,813,1.39,89.38,70822,0.54,2.68
9,D3,85,BLAST,Threshold met,1458,12.03,86.66,7940,1.77,5.10


In [5]:
# ============================================================
# 7. Inspect exactly which configurations were selected
# ============================================================

# Display the complete set of selected configurations as a table.
# This is 6 datasets × 3 thresholds × 4 methods = 72 rows.

# configuration_details = selected_results.reset_index(names='Original Index')
configuration_details = selected_results.copy()

configuration_details_display = configuration_details[
    [
        'Dataset',
        'Target Recall',
        'Method',
        'Selection',
        'Original Row Index',
        'Precision %',
        'Recall %',
        'F1 %',
        'Candidate pairs',
        'Runtime',
        'Weighting Scheme',
        'Total Runtime',
        'Configuration'
    ]
].copy()

display(
    configuration_details_display.style
    .format({
        'Precision %': '{:.2f}',
        'Recall %': '{:.2f}',
        'F1 %': '{:.2f}',
        'Candidate pairs': '{:,.0f}',
        'Runtime': '{:.2f}',
        'Total Runtime': '{:.2f}',
    }, na_rep='—')
    .hide(axis='index')
)

configuration_details_display.to_csv('../../scripts/data_manipulation/metablocking_selected_configurations.csv', index=False)

Dataset,Target Recall,Method,Selection,Original Row Index,Precision %,Recall %,F1 %,Candidate pairs,Runtime,Weighting Scheme,Total Runtime,Configuration
D2,85,WEP,Threshold met,509,12.35,85.15,21.57,"7,337",0.09,X2,1.29,"{'attributes': ['name'], 'size': 1024, 'offset': 0, 'num_hashes': 20, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D2,85,WNP,Threshold met,851,8.49,85.15,15.44,"10,673",0.28,ECBS,1.12,"{'attributes': ['name'], 'size': 512, 'offset': 0, 'num_hashes': 6, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 4}"
D2,85,CEP,Threshold met,315,3.40,85.15,6.54,"26,640",0.03,CN-CBS,0.87,"{'attributes': ['name'], 'size': 256, 'offset': 0, 'num_hashes': 2, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 2}"
D2,85,CNP,Threshold met,324,3.50,85.15,6.73,"25,866",0.25,COSINE,1.09,"{'attributes': ['name'], 'size': 256, 'offset': 0, 'num_hashes': 2, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 2}"
D2,85,BLAST,Threshold met,244,58.65,85.06,69.43,"1,543",0.49,CNC,1.50,"{'attributes': ['name'], 'size': 512, 'offset': 0, 'num_hashes': 1, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,WEP,Threshold met,812,3.65,86.39,7.00,"26,108",0.07,SN-CBS,2.21,"{'attributes': ['title'], 'size': 1024, 'offset': 0, 'num_hashes': 3, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,WNP,Threshold met,824,2.59,85.93,5.02,"36,615",0.37,X2,2.51,"{'attributes': ['title'], 'size': 1024, 'offset': 0, 'num_hashes': 3, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,CEP,Threshold met,813,1.16,89.29,2.28,"85,053",0.10,SNC,2.24,"{'attributes': ['title'], 'size': 1024, 'offset': 0, 'num_hashes': 3, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,CNP,Threshold met,813,1.39,89.38,2.74,"70,822",0.54,SNC,2.68,"{'attributes': ['title'], 'size': 1024, 'offset': 0, 'num_hashes': 3, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,BLAST,Threshold met,1458,12.03,86.66,21.12,"7,940",1.77,SNC,5.10,"{'attributes': ['title'], 'size': 256, 'offset': 0, 'num_hashes': 1, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
